# 01 – Exploratory Data Analysis
## FraudShield AI · PaySim Mobile Money Fraud Dataset

This notebook performs a comprehensive dataset audit and exploratory analysis on the
**PaySim** synthetic mobile money dataset before any modelling steps.

**Objectives:**
- Understand dataset shape, dtypes, and missing values
- Analyse class imbalance
- Profile transaction types, amounts, and time distributions
- Identify correlations and fraud-specific patterns
- Generate business-ready visualisations


In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # ensure src/ is on path

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from IPython.display import display

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#0a0e1a',
    'axes.facecolor':   '#111827',
    'axes.edgecolor':   '#334155',
    'axes.labelcolor':  '#94a3b8',
    'text.color':       '#f1f5f9',
    'xtick.color':      '#64748b',
    'ytick.color':      '#64748b',
    'grid.color':       '#1e293b',
    'grid.linestyle':   '--',
    'grid.alpha':       0.6,
    'font.family':      'DejaVu Sans',
})

print("Libraries loaded ✓")


In [ ]:
# ── 1. Load Raw Data ──────────────────────────────────────────────────────────
RAW_PATH = "data/raw/PS_20174392719_1491204439457_log.csv"

df = pd.read_csv(RAW_PATH)
print(f"Shape : {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)


## 1.1 Dataset Schema & Dtypes

In [ ]:
info_df = pd.DataFrame({
    'Column':   df.columns,
    'Dtype':    [str(d) for d in df.dtypes],
    'Non-Null': df.notna().sum().values,
    'Null':     df.isna().sum().values,
    'Unique':   [df[c].nunique() for c in df.columns],
    'Sample':   [df[c].iloc[0] for c in df.columns],
})
display(info_df.to_string(index=False))


In [ ]:
# Statistical summary (numerical columns only)
display(df.describe().T.round(4))


## 1.2 Missing Values & Duplicates

In [ ]:
print("=== Missing Values ===")
missing = df.isna().sum()
print(missing[missing > 0] if missing.any() else "None — dataset is complete ✓")

print("\n=== Duplicate Rows ===")
dup = df.duplicated().sum()
print(f"{dup:,} duplicates found")


## 1.3 Class Imbalance Analysis

In [ ]:
fraud_counts = df['isFraud'].value_counts()
fraud_pct    = fraud_counts / len(df) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(['Legitimate', 'Fraud'], fraud_counts.values,
             color=['#3b82f6', '#ef4444'], width=0.5, edgecolor='none')
for i, (cnt, pct) in enumerate(zip(fraud_counts.values, fraud_pct.values)):
    axes[0].text(i, cnt + 10_000, f"{cnt:,}\n({pct:.2f}%)",
                 ha='center', fontsize=10, color='#f1f5f9')
axes[0].set_title('Transaction Class Distribution', fontsize=13, color='#f1f5f9')
axes[0].set_ylabel('Count', color='#94a3b8')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

# Pie
wedges, texts, autotexts = axes[1].pie(
    fraud_counts.values,
    labels=['Legitimate', 'Fraud'],
    autopct='%1.3f%%',
    colors=['#3b82f6', '#ef4444'],
    startangle=140,
    wedgeprops=dict(edgecolor='#0a0e1a', linewidth=2)
)
for at in autotexts: at.set_color('#f1f5f9')
axes[1].set_title('Class Proportion', fontsize=13, color='#f1f5f9')

plt.tight_layout()
plt.savefig('reports/figures/eda_class_balance.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nImbalance ratio  —  Legit : Fraud = {fraud_counts[0]/fraud_counts[1]:.0f} : 1")


## 1.4 Transaction Type Distribution

In [ ]:
type_stats = df.groupby('type').agg(
    Count=('isFraud', 'count'),
    Fraud_Count=('isFraud', 'sum'),
).reset_index()
type_stats['Fraud_Rate_%'] = (type_stats['Fraud_Count'] / type_stats['Count'] * 100).round(4)
type_stats['Share_%']      = (type_stats['Count'] / len(df) * 100).round(2)
display(type_stats.sort_values('Fraud_Count', ascending=False).to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
colors = ['#3b82f6', '#06b6d4', '#8b5cf6', '#10b981', '#f59e0b']
bars = ax.bar(type_stats['type'], type_stats['Count'],
              color=colors[:len(type_stats)], edgecolor='none', width=0.55)

ax2 = ax.twinx()
ax2.plot(type_stats['type'], type_stats['Fraud_Rate_%'],
         'o--', color='#ef4444', linewidth=2, markersize=8, label='Fraud Rate (%)')
ax2.set_ylabel('Fraud Rate (%)', color='#ef4444')
ax2.tick_params(axis='y', colors='#ef4444')

ax.set_title('Transaction Volume & Fraud Rate by Type', fontsize=13, color='#f1f5f9')
ax.set_ylabel('Transaction Count', color='#94a3b8')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax2.legend(loc='upper right')
plt.tight_layout()
plt.savefig('reports/figures/eda_type_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


## 1.5 Amount Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Overall log-scale histogram
axes[0].hist(df['amount'].clip(upper=df['amount'].quantile(0.99)),
             bins=80, color='#3b82f6', edgecolor='none', alpha=0.85)
axes[0].set_title('Transaction Amount Distribution (99th pct clip)', fontsize=12, color='#f1f5f9')
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Count')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))

# Fraud vs Legit
for label, color, mask in [
    ('Legitimate', '#3b82f6', df['isFraud']==0),
    ('Fraud',      '#ef4444', df['isFraud']==1),
]:
    axes[1].hist(df.loc[mask, 'amount'].clip(upper=1_000_000),
                 bins=70, alpha=0.65, color=color, label=label, edgecolor='none')
axes[1].set_title('Amount: Fraud vs Legitimate', fontsize=12, color='#f1f5f9')
axes[1].set_xlabel('Amount ($)')
axes[1].legend()
axes[1].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

plt.tight_layout()
plt.savefig('reports/figures/eda_amount_dist.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Fraud   — Mean: ${df[df.isFraud==1]['amount'].mean():,.0f}  |  Median: ${df[df.isFraud==1]['amount'].median():,.0f}")
print(f"Legit   — Mean: ${df[df.isFraud==0]['amount'].mean():,.0f}  |  Median: ${df[df.isFraud==0]['amount'].median():,.0f}")


## 1.6 Temporal Analysis (Fraud Over Time)

In [ ]:
step_agg = df.groupby('step').agg(
    total=('isFraud', 'count'),
    fraud=('isFraud', 'sum')
).reset_index()
step_agg['fraud_rate'] = step_agg['fraud'] / step_agg['total'] * 100

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax1.fill_between(step_agg['step'], step_agg['total'],
                 color='#3b82f6', alpha=0.4)
ax1.plot(step_agg['step'], step_agg['total'], color='#3b82f6', linewidth=1)
ax1.set_ylabel('Transactions / hour', color='#94a3b8')
ax1.set_title('Transaction Volume Over Time (Steps = Hours)', fontsize=13, color='#f1f5f9')
ax1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))

ax2.fill_between(step_agg['step'], step_agg['fraud_rate'],
                 color='#ef4444', alpha=0.4)
ax2.plot(step_agg['step'], step_agg['fraud_rate'], color='#ef4444', linewidth=1)
ax2.set_xlabel('Step (Hour)', color='#94a3b8')
ax2.set_ylabel('Fraud Rate (%)', color='#94a3b8')
ax2.set_title('Fraud Rate Over Time', fontsize=13, color='#f1f5f9')

plt.tight_layout()
plt.savefig('reports/figures/eda_temporal.png', dpi=150, bbox_inches='tight')
plt.show()


## 1.7 Balance Anomaly Deep Dive (Fraud Patterns)

In [ ]:
# Focus on TRANSFER+CASH_OUT (fraud-susceptible types)
df_ft = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()

df_ft['balance_diff_orig'] = df_ft['oldbalanceOrg'] - df_ft['newbalanceOrig'] - df_ft['amount']
df_ft['amount_ratio']      = np.where(df_ft['oldbalanceOrg'] > 0,
                                       df_ft['amount'] / df_ft['oldbalanceOrg'], 1.0)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, title in zip(axes,
    ['balance_diff_orig', 'amount_ratio', 'oldbalanceOrg'],
    ['Balance Diff Origin', 'Amount Ratio (amount/oldBal)', 'Old Balance (Sender)']):
    for label, color, mask in [
        ('Legit', '#3b82f6', df_ft['isFraud']==0),
        ('Fraud', '#ef4444', df_ft['isFraud']==1),
    ]:
        ax.hist(df_ft.loc[mask, col].clip(
                    df_ft[col].quantile(0.01),
                    df_ft[col].quantile(0.99)),
                bins=60, alpha=0.65, color=color, label=label, edgecolor='none')
    ax.set_title(title, fontsize=11, color='#f1f5f9')
    ax.legend(fontsize=9)

plt.suptitle('Key Feature Distributions: Fraud vs Legitimate (TRANSFER + CASH_OUT)',
             fontsize=13, color='#f1f5f9', y=1.02)
plt.tight_layout()
plt.savefig('reports/figures/eda_balance_anomaly.png', dpi=150, bbox_inches='tight')
plt.show()


## 1.8 Correlation Matrix

In [ ]:
num_cols = ['step','amount','oldbalanceOrg','newbalanceOrig',
            'oldbalanceDest','newbalanceDest','isFraud']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.3f',
            cmap='RdYlBu_r', center=0, linewidths=0.5,
            linecolor='#0a0e1a', annot_kws={'size': 9},
            ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=13, color='#f1f5f9')
ax.tick_params(colors='#94a3b8')
plt.tight_layout()
plt.savefig('reports/figures/eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()


## 1.9 Key EDA Findings Summary

| Finding | Detail |
|---|---|
| **Rows** | 6,354,407 |
| **Fraud cases** | 8,213 (0.129%) |
| **Fraud types** | 100% in TRANSFER & CASH_OUT |
| **Class ratio** | ~773 legitimate per fraud transaction |
| **Balance depletion** | Fraudulent TRANSFERs drain `oldbalanceOrg` to 0 |
| **Amount ratio** | Fraud `amount/oldBal ≈ 1.0` (entire balance transferred) |
| **Receiver anomaly** | `newbalanceDest` remains 0 post-transfer in fraud cases |
| **Missing values** | None |
| **Duplicates** | None |

> **Action**: Filter to TRANSFER + CASH_OUT for modelling; engineer balance-difference features.
